# Week 6: Listing Signal Extraction Demo

This notebook demonstrates the Week 6 deliverables:

- `SignalExtractor` processing full listing records
- JSON output schema with `entities + amenities + keywords`
- Processing listing data into `data/processed/`
- Accuracy checks for structured (`>= 90%`) and free-text (`>= 75%`) targets
- Output readiness for search indexing and filtering


In [2]:
from pathlib import Path
import sys
import json
import pandas as pd

# Ensure project root is importable whether notebook is run from repo root or notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "scripts").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.signal_extractor import SignalExtractor, run_week6_pipeline

CSV_PATH = PROJECT_ROOT / "data/processed/listing_sample_cleaned.csv"
OUTPUT_PATH = PROJECT_ROOT / "data/processed/listing_signals_demo.jsonl"
LABELED_PATH = PROJECT_ROOT / "data/processed/remarks_labeled.jsonl"

CSV_PATH.exists(), LABELED_PATH.exists()

(True, True)

In [3]:
# Deliverable 1: SignalExtractor processes full listing records.
df = pd.read_csv(CSV_PATH)
extractor = SignalExtractor()
sample_record = df.iloc[0].to_dict()
sample_signals = extractor.extract_signals(sample_record)

sample_signals

{'listing_id': '1151896186',
 'entities': {'bedrooms': 2,
  'bathrooms': 1.0,
  'price': 449000,
  'sqft': [],
  'amenities': ['living space', 'spacious', 'space']},
 'amenities': ['fireplace', 'garage', 'hot_tub', 'deck'],
 'condition_keywords': [],
 'financing_terms': [],
 'location_features': [],
 'keywords': ['fireplace',
  'garage',
  'hot_tub',
  'deck',
  'living_space',
  'spacious',
  'space']}

In [4]:
# Deliverable 2-4: Run the Week 6 pipeline and output report.
report = run_week6_pipeline(
    source="csv",
    csv_path=CSV_PATH,
    output_path=OUTPUT_PATH,
    labeled_jsonl_path=LABELED_PATH,
    mysql_host="127.0.0.1",
    mysql_user="root",
    mysql_password="root",
    mysql_database="real_estate",
    mysql_port=3308,
)

report

{'source': 'csv',
 'num_records': 1000,
 'output_path': '/Users/nathanye/Documents/GitHub/nlp-internship/data/processed/listing_signals_demo.jsonl',
 'structured_accuracy': [{'metric': 'bedrooms_exact_match',
   'accuracy': 1.0,
   'numerator': 998,
   'denominator': 998},
  {'metric': 'bathrooms_exact_match',
   'accuracy': 1.0,
   'numerator': 1000,
   'denominator': 1000},
  {'metric': 'price_exact_match',
   'accuracy': 1.0,
   'numerator': 1000,
   'denominator': 1000}],
 'structured_target': 0.9,
 'structured_target_met': True,
 'free_text_accuracy': {'metric': 'free_text_amenity_match',
  'accuracy': 0.8418156808803301,
  'numerator': 612,
  'denominator': 727},
 'free_text_target': 0.75,
 'free_text_target_met': True}

In [5]:
# Validate target thresholds.
assert report["structured_target_met"] is True, "Structured target not met"
assert report["free_text_target_met"] is True, "Free-text target not met"

report["structured_accuracy"], report["free_text_accuracy"]

([{'metric': 'bedrooms_exact_match',
   'accuracy': 1.0,
   'numerator': 998,
   'denominator': 998},
  {'metric': 'bathrooms_exact_match',
   'accuracy': 1.0,
   'numerator': 1000,
   'denominator': 1000},
  {'metric': 'price_exact_match',
   'accuracy': 1.0,
   'numerator': 1000,
   'denominator': 1000}],
 {'metric': 'free_text_amenity_match',
  'accuracy': 0.8418156808803301,
  'numerator': 612,
  'denominator': 727})

In [6]:
# Deliverable 5: Output schema for indexing/filtering.
rows = []
with OUTPUT_PATH.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        rows.append(json.loads(line))
        if i >= 2:
            break

rows

[{'listing_id': '1151896186',
  'entities': {'bedrooms': 2,
   'bathrooms': 1.0,
   'price': 449000,
   'sqft': [],
   'amenities': ['living space', 'spacious', 'space']},
  'amenities': ['fireplace', 'garage', 'hot_tub', 'deck'],
  'condition_keywords': [],
  'financing_terms': [],
  'location_features': [],
  'keywords': ['fireplace',
   'garage',
   'hot_tub',
   'deck',
   'living_space',
   'spacious',
   'space']},
 {'listing_id': '1140259530',
  'entities': {'bedrooms': 4,
   'bathrooms': 3.0,
   'price': 500000,
   'sqft': [],
   'amenities': ['garage', 'spacious', 'space']},
  'amenities': ['garage'],
  'condition_keywords': [],
  'financing_terms': [],
  'location_features': ['gated'],
  'keywords': ['garage', 'gated', 'spacious', 'space']},
 {'listing_id': '1150492554',
  'entities': {'bedrooms': 4,
   'bathrooms': 5.0,
   'price': 1749900,
   'sqft': [9],
   'amenities': ['garage', 'laundry room', 'space']},
  'amenities': ['garage', 'solar', 'elevator', 'laundry'],
  'cond

In [7]:
# Create a compact search-index style table from JSONL.
index_rows = []
with OUTPUT_PATH.open("r", encoding="utf-8") as f:
    for line in f:
        rec = json.loads(line)
        index_rows.append(
            {
                "listing_id": rec["listing_id"],
                "bedrooms": rec["entities"].get("bedrooms"),
                "bathrooms": rec["entities"].get("bathrooms"),
                "price": rec["entities"].get("price"),
                "num_keywords": len(rec.get("keywords", [])),
                "keywords": ",".join(rec.get("keywords", [])),
            }
        )

index_df = pd.DataFrame(index_rows)
index_df.head()

,listing_id,bedrooms,bathrooms,price,num_keywords,keywords
0,1151896186,2.0,1.0,449000,7,"fireplace,garage,hot_tub,deck,living_space,spa..."
1,1140259530,4.0,3.0,500000,4,"garage,gated,spacious,space"
2,1150492554,4.0,5.0,1749900,10,"garage,solar,elevator,laundry,new_construction..."
3,1151473374,2.0,1.0,649000,7,"fireplace,garage,deck,updated,remodeled,turn_k..."
4,1151757929,5.0,6.0,2149000,5,"fireplace,hot_tub,laundry,laundry_room,spacious"


## Optional: Process Entire `rets_property` Table

If local MySQL is running, process the full table and save to `data/processed/rets_property_signals.jsonl`:

```python
full_report = run_week6_pipeline(
    source="mysql",
    csv_path=CSV_PATH,
    output_path=PROJECT_ROOT / "data/processed/rets_property_signals.jsonl",
    labeled_jsonl_path=LABELED_PATH,
    mysql_host="127.0.0.1",
    mysql_user="root",
    mysql_password="root",
    mysql_database="real_estate",
    mysql_port=3308,
)
full_report
```
